# Instructor Notebook 03 — NLP: Text as Numbers
**ComplianceGPT Lab · REU 2026 · Week 2**

> **Teaching script:** Run every cell top-to-bottom. All deterministic.

**Learning arc:**
1. The text representation problem: neural nets need numbers
2. Bag of Words — counts, vocabulary, limits
3. TF-IDF — reweight by rarity
4. n-grams — capturing multi-word phrases
5. **The semantic gap**: 'court order' vs 'judicial mandate' — zero similarity in TF-IDF space
6. Practical comparison on real HIPAA scenarios

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

---
## Part 1 · The Representation Problem

> **Say:** "ML algorithms work on matrices of numbers. Text is not numbers. Every NLP approach is an answer to the same question: how do we convert a string into a vector?"

> **Say:** "The quality of your answer determines how much information you preserve — and how well your downstream model performs."

In [ ]:
# Load the real HIPAA data
HIPAA_PATH = '/Users/priscilladanso/Documents/GitHub/COMPLIANCEGPT/experiments/finalserverrun/final_vast_gemma3_4b.csv'
hipaa = pd.read_csv(HIPAA_PATH)

texts  = hipaa['question'].fillna('').tolist()
labels = hipaa['ground_truth'].tolist()          # 'PERMITTED' or 'DENIED'
y      = (hipaa['ground_truth'] == 'PERMITTED').astype(int).values

print(f'Loaded {len(texts)} HIPAA court case scenarios')
print(f'Labels: {Counter(labels)}')
print()
print('Example scenario (first 300 chars):')
print('-' * 60)
print(texts[0][:300], '...')

---
## Part 2 · Bag of Words

> **Say:** "Simplest possible approach: count how many times each word appears. Ignore word order, ignore context. Each document becomes a vector of word counts."

> **Say:** "Let's build it step by step — first manually, then with sklearn."

In [ ]:
# Manual BoW on 3 toy sentences
toy = [
    "the hospital disclosed records",
    "the court ordered disclosure of records",
    "hospital records disclosed without order",
]

# Build vocabulary
vocab = sorted(set(word for doc in toy for word in doc.split()))
print('Vocabulary:', vocab)
print()

# Build BoW matrix manually
print('Bag of Words matrix:')
print(f"  {'Document':<45}  " + '  '.join(f'{w:<8}' for w in vocab))
print('  ' + '-' * (45 + len(vocab)*10))
for doc in toy:
    words = doc.split()
    counts = [words.count(w) for w in vocab]
    print(f"  {doc:<45}  " + '  '.join(f'{c:<8}' for c in counts))

print()
print('Notice: "hospital disclosed records" and "court ordered disclosure of records"')
print('look COMPLETELY different as vectors, even though both describe information disclosure.')

In [ ]:
# BoW with sklearn on real HIPAA data
bow = CountVectorizer(max_features=1000, stop_words='english')
X_bow = bow.fit_transform(texts)

print(f'BoW feature matrix: {X_bow.shape}')
print(f'  → {X_bow.shape[0]} scenarios × {X_bow.shape[1]} vocabulary words')
print(f'  → sparsity: {1 - X_bow.nnz / (X_bow.shape[0]*X_bow.shape[1]):.1%} zeros')
print()

# Top 20 most common words (after stop word removal)
word_counts = X_bow.sum(axis=0).A1
top_words = sorted(zip(bow.get_feature_names_out(), word_counts), key=lambda x: -x[1])[:20]
print('Top 20 most frequent words across all scenarios:')
for word, count in top_words:
    bar = '█' * int(count / 20)
    print(f'  {word:<20} {int(count):>5}  {bar}')

In [ ]:
# Classify HIPAA with BoW features
lr = LogisticRegression(max_iter=1000, random_state=42)
scores_bow = cross_val_score(lr, X_bow, y, cv=5, scoring='accuracy')

print('Logistic Regression + Bag of Words on HIPAA:')
print(f'  5-fold CV accuracy: {scores_bow.mean():.1%} ± {scores_bow.std():.3f}')
print()
print('Not bad — but can we do better? And why does it fail?')

---
## Part 3 · TF-IDF — Reweight by Rarity

> **Say:** "BoW counts words equally. But 'the' appears in every document — it carries zero information about PERMITTED vs DENIED. 'subpoena' appears in 3 documents — it's very informative."

> **Say:** "TF-IDF fixes this: words that are rare across documents get higher weight."

**TF-IDF formula:**
- `TF(t, d)` = count of term `t` in document `d` / total words in `d`
- `IDF(t)` = log(N / df(t)) where N = total docs, df(t) = docs containing `t`
- `TF-IDF(t, d)` = TF × IDF

In [ ]:
# Show IDF intuition
N = len(texts)
vocab_check = [
    ('the', sum(1 for t in texts if 'the' in t.lower().split())),
    ('patient', sum(1 for t in texts if 'patient' in t.lower())),
    ('court', sum(1 for t in texts if 'court' in t.lower())),
    ('subpoena', sum(1 for t in texts if 'subpoena' in t.lower())),
    ('hipaa', sum(1 for t in texts if 'hipaa' in t.lower())),
    ('judicial', sum(1 for t in texts if 'judicial' in t.lower())),
    ('abeyta', sum(1 for t in texts if 'abeyta' in t.lower())),
]

print(f'IDF intuition (N = {N} scenarios):')
print()
print(f"  {'Word':<15} {'Doc freq':>10} {'IDF':>8}  Interpretation")
print('  ' + '-' * 70)
for word, df in vocab_check:
    idf = np.log(N / max(df, 1))
    interp = 'useless (everywhere)' if idf < 1 else ('very informative' if idf > 3 else 'moderately useful')
    print(f'  {word:<15} {df:>10}  {idf:>7.2f}  {interp}')

In [ ]:
# TF-IDF with bigrams
tfidf = TfidfVectorizer(
    max_features=2000,
    stop_words='english',
    ngram_range=(1, 2)      # unigrams + bigrams
)
X_tfidf = tfidf.fit_transform(texts)

scores_tfidf = cross_val_score(lr, X_tfidf, y, cv=5, scoring='accuracy')

print(f'TF-IDF + bigrams + LogReg accuracy: {scores_tfidf.mean():.1%} ± {scores_tfidf.std():.3f}')
print(f'BoW accuracy:                       {scores_bow.mean():.1%}')
print(f'Improvement:                        +{(scores_tfidf.mean()-scores_bow.mean()):.1%}')
print()

# Show top predictive features for each class
lr_tfidf = LogisticRegression(max_iter=1000, random_state=42)
lr_tfidf.fit(X_tfidf, y)

feature_names = tfidf.get_feature_names_out()
coefs = lr_tfidf.coef_[0]

top_permitted = sorted(zip(feature_names, coefs), key=lambda x: -x[1])[:10]
top_denied    = sorted(zip(feature_names, coefs), key=lambda x:  x[1])[:10]

print('Top 10 features → PERMITTED (positive weights):')
for feat, w in top_permitted:
    print(f'  {feat:<30} {w:+.3f}')

print()
print('Top 10 features → DENIED (negative weights):')
for feat, w in top_denied:
    print(f'  {feat:<30} {w:+.3f}')

---
## Part 4 · The Semantic Gap — The Wall TF-IDF Hits

> **Say:** "TF-IDF is better than BoW, but it has a fundamental problem: it compares documents by word overlap. Two phrases that mean the same thing but use different words look completely different."

> **Say:** "For HIPAA, this is catastrophic. The law says 'court order' enables disclosure. A document might say 'judicial mandate' and mean exactly the same thing — different words, same legal effect."

> **Say:** "Watch what happens when we compute cosine similarity."

In [ ]:
# The semantic gap demonstration
legal_phrases = [
    "The judge issued a court order requiring disclosure of records",
    "A judicial mandate required the facility to produce the documents",
    "The court issued a subpoena for medical records",
    "Law enforcement requested records without any legal process",
    "The officer said they needed the patient information immediately",
    "Records were disclosed without the patient authorization form",
]

verdicts = ['PERMITTED', 'PERMITTED', 'PERMITTED', 'DENIED', 'DENIED', 'DENIED']

# TF-IDF similarity matrix
phrase_vec = TfidfVectorizer(ngram_range=(1, 2))
X_phrases = phrase_vec.fit_transform(legal_phrases)
sim_matrix = cosine_similarity(X_phrases)

print('TF-IDF Cosine Similarity Matrix (legal phrases):')
print()
labels_short = [
    'court order',
    'judicial mandate',
    'subpoena',
    'no legal process',
    'officer request',
    'no authorization',
]
print(f"  {'':22}" + ''.join(f'{s[:10]:>14}' for s in labels_short))
print('  ' + '-' * (22 + 14 * len(labels_short)))
for i, (label, verdict) in enumerate(zip(labels_short, verdicts)):
    row_vals = ''.join(f'{sim_matrix[i][j]:>14.2f}' for j in range(len(labels_short)))
    print(f'  [{verdict[:3]}] {label:<18}{row_vals}')

print()
key_pair = sim_matrix[0][1]
print(f'  KEY RESULT: "court order" ↔ "judicial mandate" similarity = {key_pair:.3f}')
print()
print('  Both phrases enable the SAME HIPAA exception.')
print('  TF-IDF says they are {:.0%} different. That is the semantic gap.'.format(1-key_pair))

In [ ]:
# Find real HIPAA scenario pairs where TF-IDF misclassifies
# Show a false negative: true PERMITTED but predicted DENIED
lr_fitted = LogisticRegression(max_iter=1000, random_state=42)
lr_fitted.fit(X_tfidf, y)
preds = lr_fitted.predict(X_tfidf)

false_negatives = [i for i, (truth, pred) in enumerate(zip(y, preds)) if truth == 1 and pred == 0]
false_positives = [i for i, (truth, pred) in enumerate(zip(y, preds)) if truth == 0 and pred == 1]

print(f'TF-IDF + LogReg on training set:')
print(f'  False Negatives (predicted DENIED, truth PERMITTED): {len(false_negatives)}')
print(f'  False Positives (predicted PERMITTED, truth DENIED): {len(false_positives)}')
print()
if false_negatives:
    idx = false_negatives[0]
    print(f'Example False Negative (case {hipaa["row_id"].iloc[idx]}):')
    print(f'  Ground truth:  PERMITTED')
    print(f'  TF-IDF pred:   DENIED')
    print(f'  Text: {texts[idx][:250]}...')
    print()
    print('  → This case likely uses a legal synonym or indirect phrasing.')
    print('  → TF-IDF does not understand meaning — only word overlap.')

---
## Part 5 · Full Comparison So Far

> **Say:** "Let's put all the text representation approaches side by side and see where we stand."

In [ ]:
# Quick BoW-only classifier as baseline
lr_bow = LogisticRegression(max_iter=1000, random_state=42)
scores_bow_only = cross_val_score(lr_bow, X_bow, y, cv=5)

results = [
    ('Majority class baseline',          max(y.mean(), 1-y.mean()),     'Always predict most common class'),
    ('Bag of Words + LogReg',            scores_bow_only.mean(),        '1000 word count features'),
    ('TF-IDF (unigrams+bigrams) + LogReg', scores_tfidf.mean(),        '2000 TF-IDF features'),
    ('LLM extraction + formal engine',   0.9416,                        'Gemma3-4B + Soufflé (our system)'),
]

print('HIPAA compliance classification — method comparison:')
print()
print(f"  {'Method':<40} {'Accuracy':>10}  Notes")
print('  ' + '-' * 80)
for method, acc, notes in results:
    bar = '█' * int(acc * 30)
    print(f"  {method:<40} {acc:>9.1%}  {notes}")

print()
print('  Gap between TF-IDF and LLM+engine: {:.1%}'.format(0.9416 - scores_tfidf.mean()))
print()
print('  → Why does LLM extraction win?')
print('     1. It understands synonyms ("judicial mandate" = "court order")')
print('     2. It extracts structured predicates, not just word patterns')
print('     3. The formal engine applies exact legal logic on clean predicates')
print('  → But: the LLM still makes 8 mistakes. Those are your research target.')

---
## Bonus · N-grams — Capturing Phrases

> **Say:** "One improvement over plain BoW: n-grams. Instead of just single words (unigrams), also count pairs (bigrams) or triples (trigrams). 'court order' as a bigram is more informative than 'court' and 'order' separately."

In [ ]:
# Show what ngrams adds
sentence = "the hospital disclosed records pursuant to a court order"
print(f'Sentence: "{sentence}"')
print()

for n, label in [(1,'Unigrams'), (2,'Bigrams'), (3,'Trigrams')]:
    vec = CountVectorizer(ngram_range=(n,n))
    vec.fit([sentence])
    terms = vec.get_feature_names_out()
    print(f'{label} ({len(terms)}): {", ".join(repr(t) for t in terms[:8])}')
    if len(terms) > 8:
        print(f'  ... and {len(terms)-8} more')

print()
print('"court order" as a bigram captures the legal phrase as a unit.')
print('But it still won\'t match "judicial mandate" — different words.')
print('For that, we need embeddings (next notebook).')

---
## Summary

| Approach | Key idea | Accuracy | Fatal flaw |
|---|---|---|---|
| **Bag of Words** | Count word occurrences | ~70–80% | No word order, no meaning |
| **TF-IDF** | Rare words weighted more | ~80–85% | Still no semantic similarity |
| **n-grams** | Capture multi-word phrases | +1–3% | Same flaw, bigger vocabulary |
| **Need: Embeddings** | Meaning as geometry | ? | Next session |

**The core problem:** TF-IDF measures *word overlap*, not *meaning*. Two phrases that mean the same thing but use different words have zero similarity.

> **Transition:** "The solution is word embeddings. Instead of sparse word-count vectors, we learn dense vectors where semantically similar phrases are geometrically close. 'Court order' and 'judicial mandate' end up at distance 0.13 instead of 0.92. That's the next session."